In [1]:
# =======================================================================
# BAGIAN 1: SETUP DAN IMPOR LIBRARY
# =======================================================================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import libpysal.weights
import esda
import spreg
import mgwr
import statsmodels.api as sm
from splot.esda import plot_moran, lisa_cluster

In [2]:
# =======================================================================
# BAGIAN 2: MEMUAT DATA DAN PREPROCESSING NAMA PROVINSI
# =======================================================================

# --- Menentukan Path ke File Sumber ---
path_to_excel = "C:/Uner/Lomba/Smatic/Dataset/Data Smatic.xlsx"
path_to_shp = "C:/Uner/Semester 4/Analisis Data Spasial/UTS/gadm41_IDN_1.shp"

# --- Memuat Data ---
try:
    df_data = pd.read_excel(path_to_excel, sheet_name="Sheet1")
    gdf = gpd.read_file(path_to_shp)
except Exception as e:
    print(f"!!! ERROR saat memuat file awal: {e} !!!")
    exit()

# --- LANGKAH 1: DETEKSI NAMA PROVINSI YANG TIDAK COCOK ---
# Kita cari tahu nama apa saja yang ada di satu file, tapi tidak ada di file lainnya.
provinsi_peta = set(gdf['NAME_1'])
provinsi_excel = set(df_data['Provinsi'])

unmatched_in_excel = sorted(list(provinsi_excel - provinsi_peta))
unmatched_in_peta = sorted(list(provinsi_peta - provinsi_excel))

print("--- DETEKSI NAMA PROVINSI YANG BERBEDA ---")
if unmatched_in_excel or unmatched_in_peta:
    print("\nDitemukan ketidakcocokan nama provinsi. Anda perlu menyamakannya.")
    print("\nNama Provinsi yang ada di Excel TAPI TIDAK ADA di Peta:")
    print(unmatched_in_excel)
    print("\nNama Provinsi yang ada di Peta TAPI TIDAK ADA di Excel:")
    print(unmatched_in_peta)
else:
    print("Selamat! Semua nama provinsi sudah cocok. Tidak perlu preprocessing.")

# --- LANGKAH 2: BUAT KAMUS PEMETAAN UNTUK MEMPERBAIKI NAMA ---
# Kunci kamus adalah nama yang salah/berbeda, dan value adalah nama yang benar/standar.
# Saya sudah isikan beberapa perbaikan umum.
# LENGKAPI KAMUS INI berdasarkan daftar ketidakcocokan yang muncul di atas.

province_mapping = {
    # Kemungkinan salah ketik atau singkatan di file Excel Anda
    'DKI Jakarta': 'Daerah Khusus Ibukota Jakarta',
    'D.I. Yogyakarta': 'Daah Istimewa Yogyakarta',
    'Yogyakarta': 'Daerah Istimewa Yogyakarta',
    'Jabar': 'Jawa Barat',
    'Jateng': 'Jawa Tengah',
    'Jatim': 'Jawa Timur',
    'Babel': 'Kepulauan Bangka Belitung',
    'Kep. Riau': 'Kepulauan Riau',
    
    # Tambahkan perbaikan Anda sendiri di sini. Contoh:
    # 'Nama Salah di Excel': 'Nama Benar Sesuai Peta',
    
}

# --- LANGKAH 3: APLIKASIKAN PEMETAAN DAN GABUNGKAN DATA ---
# Buat kolom baru dengan nama yang sudah distandarkan di kedua dataframe
df_data['provinsi_std'] = df_data['Provinsi'].replace(province_mapping)
gdf['provinsi_std'] = gdf['NAME_1'].replace(province_mapping)

# Gabungkan data menggunakan kolom baru yang sudah bersih
gdf_merged = gdf.merge(df_data, on='provinsi_std', how='inner', suffixes=('_peta', '_excel'))

# --- VALIDASI HASIL PENGGABUNGAN ---
if gdf_merged.empty:
    print("\n!!! ERROR FATAL: Penggabungan GAGAL bahkan setelah preprocessing. !!!")
    print("Pastikan Anda sudah melengkapi 'province_mapping' dengan benar.")
    exit()
else:
    print(f"\nPreprocessing Berhasil! {len(gdf_merged)} provinsi berhasil digabungkan.")

--- DETEKSI NAMA PROVINSI YANG BERBEDA ---

Ditemukan ketidakcocokan nama provinsi. Anda perlu menyamakannya.

Nama Provinsi yang ada di Excel TAPI TIDAK ADA di Peta:
['ACEH', 'BALI', 'BANTEN', 'BENGKULU', 'DI YOGYAKARTA', 'DKI JAKARTA', 'GORONTALO', 'JAMBI', 'JAWA BARAT', 'JAWA TENGAH', 'JAWA TIMUR', 'KALIMANTAN BARAT', 'KALIMANTAN SELATAN', 'KALIMANTAN TENGAH', 'KALIMANTAN TIMUR', 'KALIMANTAN UTARA', 'KEP. BANGKA BELITUNG', 'KEP. RIAU', 'LAMPUNG', 'MALUKU', 'MALUKU UTARA', 'NUSA TENGGARA BARAT', 'NUSA TENGGARA TIMUR', 'PAPUA', 'PAPUA BARAT', 'RIAU', 'SULAWESI BARAT', 'SULAWESI SELATAN', 'SULAWESI TENGAH', 'SULAWESI TENGGARA', 'SULAWESI UTARA', 'SUMATERA BARAT', 'SUMATERA SELATAN', 'SUMATERA UTARA']

Nama Provinsi yang ada di Peta TAPI TIDAK ADA di Excel:
['Aceh', 'Bali', 'Bangka Belitung', 'Banten', 'Bengkulu', 'Gorontalo', 'Jakarta Raya', 'Jambi', 'Jawa Barat', 'Jawa Tengah', 'Jawa Timur', 'Kalimantan Barat', 'Kalimantan Selatan', 'Kalimantan Tengah', 'Kalimantan Timur', 'Kalimantan

In [3]:
# =======================================================================
# BAGIAN 3: DEFINISIKAN VARIABEL & BUAT MATRIKS PEMBOBOT
# =======================================================================

# --- Definisikan Variabel untuk Analisis ---
y_name = 'y'
x_names = ['x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8']

try:
    y = gdf_merged[y_name]
    X = gdf_merged[x_names]
except KeyError as e:
    print(f"\n!!! ERROR: Kolom {e} tidak ditemukan. Pastikan nama kolom di file Excel Anda sudah benar. !!!")
    exit()

X = sm.add_constant(X)
print("\n--- Variabel untuk analisis berhasil ditentukan. ---")

# --- Membuat Matriks Pembobot Spasial (w) ---
try:
    print("\nMembuat Matriks Pembobot Spasial (W)...")
    w = libpysal.weights.Queen.from_dataframe(gdf_merged)
    w.transform = 'r' 
    print("Matriks W (Queen Contiguity) berhasil dibuat.")
    
    # Visualisasi
    fig, ax = plt.subplots(figsize=(10, 10))
    gdf_merged.plot(edgecolor='grey', facecolor='w', ax=ax)
    w.plot(gdf_merged, ax=ax, edge_kws=dict(color='red', linestyle=':', linewidth=1), node_kws=dict(marker=''))
    ax.set_title("Peta Konektivitas Tetangga (Queen Contiguity)")
    plt.show()

except Exception as e:
    print(f"\n!!! ERROR saat membuat Matriks Pembobot Spasial: {e} !!!")


--- Variabel untuk analisis berhasil ditentukan. ---

Membuat Matriks Pembobot Spasial (W)...

!!! ERROR saat membuat Matriks Pembobot Spasial:  !!!


C:\Users\Faiz Iqbal\AppData\Local\Temp\ipykernel_32236\4012567689.py:22: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = libpysal.weights.Queen.from_dataframe(gdf_merged)


In [4]:

# --- Membuat Matriks Pembobot Spasial (w) ---
# Ini adalah bagian yang membuat variabel 'w'
try:
    print("\nMembuat Matriks Pembobot Spasial (W)...")
    w = libpysal.weights.Queen.from_dataframe(gdf_merged)
    w.transform = 'r' 
    
    print(f"Matriks W (Queen Contiguity) berhasil dibuat.")
    
    # Visualisasi konektivitas tetangga
    fig, ax = plt.subplots(figsize=(10, 10))
    gdf_merged.plot(edgecolor='grey', facecolor='w', ax=ax)
    w.plot(gdf_merged, ax=ax,
           edge_kws=dict(color='red', linestyle=':', linewidth=1),
           node_kws=dict(marker=''))
    ax.set_title("Peta Konektivitas Tetangga (Queen Contiguity)")
    plt.show()

except Exception as e:
    print(f"\n!!! ERROR saat membuat Matriks Pembobot Spasial: {e} !!!")



Membuat Matriks Pembobot Spasial (W)...

!!! ERROR saat membuat Matriks Pembobot Spasial:  !!!


C:\Users\Faiz Iqbal\AppData\Local\Temp\ipykernel_32236\2602443861.py:5: FutureWarning: `use_index` defaults to False but will default to True in future. Set True/False directly to control this behavior and silence this warning
  w = libpysal.weights.Queen.from_dataframe(gdf_merged)


In [ ]:
# =======================================================================
# BAGIAN 4: ANALISIS DEPENDENSI SPASIAL GLOBAL (MORAN'S I)
# =======================================================================
# Kode ini sekarang akan berjalan karena variabel 'y' dan 'w' sudah ada.

# Menghitung Moran's I untuk variabel dependen (y)
moran = esda.Moran(y, w)

print("\n--- Hasil Uji Moran's I Global ---")
print(f"Nilai Moran's I : {moran.I:.4f}")
print(f"Nilai p-value   : {moran.p_sim:.4f} (berdasarkan 999 simulasi)")
print(f"Nilai z-score   : {moran.z_sim:.4f} (berdasarkan 999 simulasi)")

# Interpretasi Hasil
if moran.p_sim < 0.05:
    print("\nInterpretasi: p-value signifikan (< 0.05).")
    if moran.I > 0:
        print("Terdapat OTOKORELASI SPASIAL POSITIF.")
    else:
        print("Terdapat OTOKORELASI SPASIAL NEGATIF.")
else:
    print("\nInterpretasi: p-value tidak signifikan. Sebaran nilai 'y' cenderung acak.")

# Visualisasi dengan Moran Scatter Plot
fig, ax = plt.subplots(figsize=(7, 7))
plot_moran(moran, ax=ax)
ax.set_title("Moran's I Scatter Plot untuk Variabel Dependen 'y'")
plt.show()

NameError: name 'w' is not defined

: 